<a href="https://colab.research.google.com/github/Naveenyadav60000/AI-ML/blob/main/Day_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Problem Statement
Tune & Ensemble a Classifier — Take a baseline classifier (e.g. Decision Tree or Logistic Regression), evaluate it with 5-fold Cross Validation, then tune it with GridSearchCV AND RandomizedSearchCV. Finally, build one Ensemble model (Random Forest, Bagging or Voting Classifier) and compare all results in a single table.
Time limit: 60 minutes
Deliverables
Notebook showing baseline CV score, GridSearchCV and RandomizedSearchCV best params/scores
One ensemble model trained and evaluated on the same split
A single results table: Model | CV Accuracy | Test Accuracy | Best Params


https://www.kaggle.com/datasets/mahatiratusher/heart-disease-risk-prediction-dataset

import libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score

Load the dataset

In [ ]:
df = pd.read_csv("heart_disease_risk_dataset_earlymed.csv")

df.head()

,Chest_Pain,Shortness_of_Breath,Fatigue,Palpitations,Dizziness,Swelling,Pain_Arms_Jaw_Back,Cold_Sweats_Nausea,High_BP,High_Cholesterol,Diabetes,Smoking,Obesity,Sedentary_Lifestyle,Family_History,Chronic_Stress,Gender,Age,Heart_Risk
0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,48.0,0.0
1,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,46.0,0.0
2,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,66.0,0.0
3,1.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,60.0,1.0
4,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,69.0,0.0


Check the data

In [ ]:
print(df.shape)
print(df.columns)
print(df.isnull().sum())

(70000, 19)
Index(['Chest_Pain', 'Shortness_of_Breath', 'Fatigue', 'Palpitations',
       'Dizziness', 'Swelling', 'Pain_Arms_Jaw_Back', 'Cold_Sweats_Nausea',
       'High_BP', 'High_Cholesterol', 'Diabetes', 'Smoking', 'Obesity',
       'Sedentary_Lifestyle', 'Family_History', 'Chronic_Stress', 'Gender',
       'Age', 'Heart_Risk'],
      dtype='object')
Chest_Pain             0
Shortness_of_Breath    0
Fatigue                0
Palpitations           0
Dizziness              0
Swelling               0
Pain_Arms_Jaw_Back     0
Cold_Sweats_Nausea     0
High_BP                0
High_Cholesterol       0
Diabetes               0
Smoking                0
Obesity                0
Sedentary_Lifestyle    0
Family_History         0
Chronic_Stress         0
Gender                 0
Age                    0
Heart_Risk             0
dtype: int64


Separate X and y

First, find the target column

In [ ]:
df.columns

Index(['Chest_Pain', 'Shortness_of_Breath', 'Fatigue', 'Palpitations',
       'Dizziness', 'Swelling', 'Pain_Arms_Jaw_Back', 'Cold_Sweats_Nausea',
       'High_BP', 'High_Cholesterol', 'Diabetes', 'Smoking', 'Obesity',
       'Sedentary_Lifestyle', 'Family_History', 'Chronic_Stress', 'Gender',
       'Age', 'Heart_Risk'],
      dtype='object')

In [ ]:
X = df.drop("Heart_Risk", axis=1)
y = df["Heart_Risk"]

Convert categorical columns

In [ ]:
X = pd.get_dummies(X, drop_first=True)

Now split the data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Baseline Decision Tree

In [ ]:
baseline_model = DecisionTreeClassifier(random_state=42)

baseline_cv = cross_val_score(
    baseline_model,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy"
)

print("5-Fold CV Accuracy:", baseline_cv.mean())

5-Fold CV Accuracy: 0.9815178571428571


Train on training data

In [ ]:
baseline_model.fit(X_train, y_train)

baseline_test = baseline_model.predict(X_test)

baseline_test_accuracy = accuracy_score(
    y_test,
    baseline_test
)

print("Test Accuracy:", baseline_test_accuracy)

Test Accuracy: 0.9817857142857143


GridSearchCV

In [ ]:
param_grid = {
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

Create GridSearch

In [ ]:
grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="accuracy"
)

Train

In [ ]:
grid_search.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(random_state=42),
             param_grid={'max_depth': [3, 5, 7, 10, None],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10]},
             scoring='accuracy')

Best parameters

In [ ]:
print("Best Parameters:")
print(grid_search.best_params_)

Best Parameters:
{'max_depth': None, 'min_samples_leaf': 4, 'min_samples_split': 10}


Best CV score

In [ ]:
print("Best CV Accuracy:")
print(grid_search.best_score_)

Best CV Accuracy:
0.9825178571428571


Test accuracy

In [ ]:
grid_pred = grid_search.predict(X_test)

grid_test_accuracy = accuracy_score(
    y_test,
    grid_pred
)

print("GridSearch Test Accuracy:", grid_test_accuracy)

GridSearch Test Accuracy: 0.9835


RandomizedSearchCV

In [ ]:
random_search = RandomizedSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    n_iter=10,
    cv=5,
    scoring="accuracy",
    random_state=42
)

Train

In [ ]:
random_search.fit(X_train, y_train)

RandomizedSearchCV(cv=5, estimator=DecisionTreeClassifier(random_state=42),
                   param_distributions={'max_depth': [3, 5, 7, 10, None],
                                        'min_samples_leaf': [1, 2, 4],
                                        'min_samples_split': [2, 5, 10]},
                   random_state=42, scoring='accuracy')

Best parameters

In [ ]:
print("Best Parameters:")
print(random_search.best_params_)

Best Parameters:
{'min_samples_split': 5, 'min_samples_leaf': 4, 'max_depth': None}


Best CV score

In [ ]:
print("Best CV Accuracy:")
print(random_search.best_score_)

Best CV Accuracy:
0.9823928571428571


Test accuracy

In [ ]:
random_pred = random_search.predict(X_test)

random_test_accuracy = accuracy_score(
    y_test,
    random_pred
)

print("RandomizedSearch Test Accuracy:", random_test_accuracy)

RandomizedSearch Test Accuracy: 0.9832142857142857


Ensemble Model — Random Forest

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

5-fold CV

In [ ]:
rf_cv = cross_val_score(
    rf_model,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy"
)

print("Random Forest CV Accuracy:", rf_cv.mean())

Random Forest CV Accuracy: 0.992


Train:

In [ ]:
rf_model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

Predict:

In [ ]:
rf_pred = rf_model.predict(X_test)

Test accuracy

In [ ]:
rf_test_accuracy = accuracy_score(
    y_test,
    rf_pred
)

print("Random Forest Test Accuracy:", rf_test_accuracy)

Random Forest Test Accuracy: 0.9916428571428572


Final Results Table

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Baseline Decision Tree",
        "GridSearch Decision Tree",
        "RandomizedSearch Decision Tree",
        "Random Forest"
    ],

    "CV Accuracy": [
        baseline_cv.mean(),
        grid_search.best_score_,
        random_search.best_score_,
        rf_cv.mean()
    ],

    "Test Accuracy": [
        baseline_test_accuracy,
        grid_test_accuracy,
        random_test_accuracy,
        rf_test_accuracy
    ],

    "Best Params": [
        "Default",
        grid_search.best_params_,
        random_search.best_params_,
        {
            "n_estimators": 100
        }
    ]
})

results

,Model,CV Accuracy,Test Accuracy,Best Params
0,Baseline Decision Tree,0.981518,0.981786,Default
1,GridSearch Decision Tree,0.982518,0.983500,"{'max_depth': None, 'min_samples_leaf': 4, 'mi..."
2,RandomizedSearch Decision Tree,0.982393,0.983214,"{'min_samples_split': 5, 'min_samples_leaf': 4..."
3,Random Forest,0.992000,0.991643,{'n_estimators': 100}
